# 模型
大型语言模型（LLM）是强大的 AI 工具，能够像人类一样解释和生成文本。它们功能多样，可以撰写内容、翻译语言、总结和回答问题，而无需针对每个任务进行专门训练。

除了文本生成，许多模型还支持：
- 工具调用 - 调用外部工具（如数据库查询或 API 调用）并在其响应中使用结果。
- 结构化输出 - 模型响应被限制为遵循定义的格式。
- 多模态 - 处理和返回除文本之外的数据，例如图像、音频和视频。
- 推理 - 模型执行多步推理以得出结论。

模型是智能体的推理引擎。它们驱动代理的决策过程，确定调用哪些工具、如何解释结果以及何时提供最终答案。

选择的模型的质量和能力直接影响代理的可靠性和性能。不同的模型擅长不同的任务——有些更擅长遵循复杂指令，有些更擅长结构化推理，有些支持更大的上下文窗口来处理更多信息。

LangChain 的标准模型接口让可以访问许多不同的提供商集成，这使得实验和切换模型变得容易，从而找到最适合情况的模型。

## 1 基本用法
模型可以通过两种方式使用
- 与代理一起使用 - 在创建代理时可以动态指定模型。
- 独立使用 - 模型可以直接调用（在代理循环之外），用于文本生成、分类或提取等任务，而无需代理框架。

相同的模型接口在两种上下文中都适用，这可以灵活地从简单开始，并根据需要扩展到更复杂的基于代理的工作流。

### 1.1 初始化模型
在 LangChain 中使用独立模型最简单的方法是使用 init_chat_model 从选择的聊天模型提供商初始化一个模型（示例如下）

In [4]:
# 没有钱所以实际用langchain_ollama
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "sk-..."

model = init_chat_model("gpt-4.1")

### 1.2 参数
聊天模型接受可用于配置其行为的参数。支持的完整参数集因模型和提供商而异，但标准参数包括

`model`
str必填
希望与提供商一起使用的特定模型的名称或标识符。

`api_key`
str
用于与模型提供商进行身份验证所需的密钥。通常在您注册访问模型时颁发。通常通过设置环境变量来访问。环境变量.

`temperature`
int
控制模型输出的随机性。数字越高，响应越有创意；数字越低，响应越确定。

`timeout`
int
在取消请求之前，等待模型响应的最长时间（以秒为单位）。

`max_tokens`
int
限制响应中token的总数，有效控制输出的长度。token在响应中，有效控制输出的长度。

`max_retries`
int
如果请求因网络超时或速率限制等问题而失败，系统将尝试重新发送请求的最大次数。

使用init_chat_model，将这些参数作为内联**kwargs:

In [ ]:
model = init_chat_model(
    "claude-sonnet-4-5-20250929",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
)

In [5]:
from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3:0.6b",
    # Kwargs passed to the model:
    temperature=0.7,
    # timeout=30,
    # max_tokens=1000,
)

## 2. 调用
必须调用聊天模型才能生成输出。有三种主要的调用方法，每种都适用于不同的用例。
### 2.1 调用
调用模型的直接方法是使用`invoke()`，带一个消息或一个消息列表。

In [21]:
response = model.invoke("Why do parrots have colorful feathers? Limit within 10 words")
print(response)

content='Parrots have colorful feathers for communication and to attract mates, which are natural traits of their species.' additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:56:58.9952757Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2658527600, 'load_duration': 65199800, 'prompt_eval_count': 24, 'prompt_eval_duration': 24021700, 'eval_count': 111, 'eval_duration': 2537225100, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'} id='lc_run--019bbb14-1d8d-71d0-8311-e0f553036b28-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 111, 'total_tokens': 135}


可以向模型提供消息列表来表示对话历史记录。每条消息都有一个角色，模型使用该角色来指示对话中谁发送了消息。有关角色、类型和内容的更多详细信息，请参阅消息指南。

In [8]:
# 字典格式
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content='Je adore créer des applications.' additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:15:24.1090676Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6936505600, 'load_duration': 79206100, 'prompt_eval_count': 57, 'prompt_eval_duration': 39930300, 'eval_count': 268, 'eval_duration': 6734594300, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'} id='lc_run--019bbaed-fb2e-7710-8289-8702d9b7e243-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 57, 'output_tokens': 268, 'total_tokens': 325}


In [9]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content="J'adore construire des applications." additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:15:32.9090541Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7500344500, 'load_duration': 79038700, 'prompt_eval_count': 57, 'prompt_eval_duration': 25277000, 'eval_count': 298, 'eval_duration': 7318894900, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'} id='lc_run--019bbaee-1b5f-7fa0-88f4-f779840143a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 57, 'output_tokens': 298, 'total_tokens': 355}


### 2.2 流式处理
大多数模型可以在生成输出内容时进行流式传输。通过逐步显示输出，流式传输显著改善了用户体验，特别是对于较长的响应。
调用 `stream()` 返回一个迭代器，该迭代器在生成输出块时逐个生成。可以使用循环实时处理每个块：

In [20]:
for chunk in model.stream("Why do parrots have colorful feathers? Limit within 10 words"):
    print(chunk.text, end="|", flush=True)

|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||Par|rots| have| colorful| feathers| for| camouflage| and| communication|,| which| helps| them| survive| and| thrive| in| their| natural| habitats|.|||

与`invoke()`不同，`invoke()`在模型完成生成完整响应后返回单个`AIMessage`，而`stream()`返回多个`AIMessageChunk`对象，每个对象包含输出文本的一部分。重要的是，流中的每个块都可以通过求和聚合为完整的消息。

In [17]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky? Limit within 10 words"):
    full = chunk if full is None else full + chunk
    # print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]

[{'type': 'text', 'text': 'The sky is a deep blue.'}]


生成的letter可以像使用invoke()生成的letter一样处理——例如，它可以聚合到letter历史记录中，并作为会话上下文传回给模型。
### 2.3 批量处理
批量处理对模型的独立请求集合可以显著提高性能并降低成本，因为处理可以并行完成。

In [19]:
responses = model.batch([
    "Why do parrots have colorful feathers? Limit within 10 words",
    "How do airplanes fly? Limit within 10 words",
    "What is quantum computing? Limit within 10 words"
])
for response in responses:
    print(response)

content='Parrots have colorful feathers for communication and attracting mates.' additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:56:23.0146791Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9815205700, 'load_duration': 96273900, 'prompt_eval_count': 24, 'prompt_eval_duration': 190340300, 'eval_count': 102, 'eval_duration': 2315042000, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'} id='lc_run--019bbb13-7507-7320-afde-0e65230e011a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 102, 'total_tokens': 126}
content='Airplanes fly by using lift and thrust from engines.' additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:56:20.4838936Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7284428800, 'load_duration': 93604400, 'prompt_eval_count': 21, 'prompt_eval_duration': 158957500, 'eval_count': 170, 'eval_duration': 37805

默认情况下，`batch()` 将仅返回整个批次的最终输出。如果希望在每个单独的输入生成完成后接收其输出，可以使用 `batch_as_completed()` 进行流式传输。

In [22]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers? Limit within 10 words",
    "How do airplanes fly? Limit within 10 words",
    "What is quantum computing? Limit within 10 words"
]):
    print(response)

(0, AIMessage(content='Parrots have colorful feathers to communicate with other parrots and to adapt their environment (e.g., hunting or nesting).', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:59:42.8549425Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2917113500, 'load_duration': 124312200, 'prompt_eval_count': 24, 'prompt_eval_duration': 34185600, 'eval_count': 120, 'eval_duration': 2733323000, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bbb16-9c9b-71e1-9624-8c116b5be99f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 120, 'total_tokens': 144}))
(1, AIMessage(content='Airplanes fly by using lift and thrust.', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T05:59:46.0695513Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6131711300, 'load_duration': 124838500, 'prompt_eval_count': 2

## 3. 工具调用
模型可以请求调用工具来执行任务，例如从数据库中获取数据、搜索网页或运行代码。工具是以下内容的组合：
- 一个 schema，包括工具名称、描述和/或参数定义（通常是 JSON schema）
- 要执行的函数或协程。协程要执行。

要使您定义的工具可供模型使用，您必须使用 `bind_tools()` 绑定它们。在随后的调用中，模型可以根据需要选择调用任何绑定的工具。

一些模型提供商提供内置工具，可以通过模型或调用参数启用（例如 ChatOpenAI，ChatAnthropic）。请查看相应的提供商参考以获取详细信息。

In [23]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


当绑定用户定义的工具时，模型的响应会包含执行工具的请求。当模型独立于智能体使用时，由用户来执行请求的操作并将结果返回给模型，以便在后续推理中使用。请注意，当使用智能体时，智能体循环将处理工具执行循环。

### 3.1 工具执行循环
当模型返回工具调用时，需要执行这些工具并将结果传回给模型。这会创建一个对话循环，模型可以使用工具结果来生成最终响应。LangChain 包含agent抽象，可以处理这种编排。

In [27]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
# print(ai_msg)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # print(tool_call)
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    # print(tool_result)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston. 🌤️ Would you like any additional information about the weather?


工具返回的每个ToolMessage都包含一个与原始工具调用匹配的tool_call_id，帮助模型将结果与请求关联起来。
### 3.2 强制工具调用
默认情况下，模型可以根据用户的输入自由选择使用哪个绑定工具。但是，可能希望强制选择一个工具，确保模型使用特定工具或给定列表中的任何工具。

In [28]:
@tool
def tool_1(input: str) -> str:
    """Tool 1"""
    return f"Tool 1: {input}"

model_with_tools = model.bind_tools([tool_1], tool_choice="any")

In [29]:
model_with_tools = model.bind_tools([tool_1], tool_choice="tool_1")

### 3.3 并行工具调用
许多模型在适当时支持并行调用多个工具。这允许模型同时从不同来源收集信息。

In [30]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "What's the weather in Boston and Tokyo?"
)


# The model may generate multiple tool calls
print(response.tool_calls)
# [
#   {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1'},
#   {'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': 'call_2'},
# ]


# Execute all tools (can be done in parallel with async)
results = []
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_weather':
        result = get_weather.invoke(tool_call)
    ...
    results.append(result)

[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '8304cb51-6a10-4871-9d82-528944bf3890', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': '77d48876-7d36-42f9-b48e-6ba0cd2f73cd', 'type': 'tool_call'}]


### 3.4 流式工具调用
流式传输响应时，工具调用通过`ToolCallChunk`逐步构建。这允许您在生成工具调用时查看它们，而不是等待完整的响应。

In [31]:
for chunk in model_with_tools.stream(
    "What's the weather in Boston and Tokyo?"
):
    # Tool call chunks arrive progressively
    for tool_chunk in chunk.tool_call_chunks:
        if name := tool_chunk.get("name"):
            print(f"Tool: {name}")
        if id_ := tool_chunk.get("id"):
            print(f"ID: {id_}")
        if args := tool_chunk.get("args"):
            print(f"Args: {args}")

Tool: get_weather
ID: d36ff170-6947-42de-88a5-be445b117c9f
Args: {"location": "Boston"}
Tool: get_weather
ID: 5483d8c9-b0d1-4e9c-8b19-457f1acbe7ff
Args: {"location": "Tokyo"}


In [33]:
gathered = None
for chunk in model_with_tools.stream("What's the weather in Boston?"):
    gathered = chunk if gathered is None else gathered + chunk
    print(gathered.tool_calls)

## 4. 结构化输出
可以要求模型以符合给定模式的格式提供其响应。这对于确保输出能够轻松解析并在后续处理中使用非常有用。LangChain 支持多种模式类型和强制结构化输出的方法。
### 4.1 Pydantic
Pydantic 模型提供最丰富的功能集，包括字段验证、描述和嵌套结构。

In [34]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

title='Inception' year=2010 director='Christopher Nolan' rating=8.5


### 4.2 TypedDict
Python 的 TypedDict 提供了一个比 Pydantic 模型更简单的替代方案，当不需要运行时验证时，这是一个理想的选择。

In [35]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.75}


### 4.3 JSON Schema
提供 JSON Schema 以实现最大控制和互操作性。

In [36]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.5}


### 4.4 示例
#### 4.4.1 消息输出与解析结构并存
将原始 `AIMessage` 对象与解析后的表示一起返回可能很有用，以便访问响应元数据，例如`token` 计数。为此，在调用 `with_structured_output` 时设置 `include_raw=True`

In [37]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
# {
#     "raw": AIMessage(...),
#     "parsed": Movie(title=..., year=..., ...),
#     "parsing_error": None,
# }

{'raw': AIMessage(content='{ "title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.5 }', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-14T08:32:26.2161199Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12772986300, 'load_duration': 859612800, 'prompt_eval_count': 305, 'prompt_eval_duration': 2551314500, 'eval_count': 33, 'eval_duration': 957026600, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bbba2-4881-71f1-bde7-3e986f0a64f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 305, 'output_tokens': 33, 'total_tokens': 338}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5),
 'parsing_error': None}

#### 4.4.2 嵌套结构

In [38]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

In [39]:
from typing_extensions import Annotated, TypedDict

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, ..., "Budget in millions USD"]

model_with_structure = model.with_structured_output(MovieDetails)

## 4. 高级主题
### 4.1 多模态
某些模型可以处理并返回非文本数据，例如图像、音频和视频。您可以通过提供内容块将非文本数据传递给模型。

有些模型可以将多模态数据作为响应的一部分返回。如果被调用以这样做，生成的AIMessage将具有多模态类型的内容块。

In [ ]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="qwen3-vl:2b")
response = model.invoke("Create a picture of a cat")
print(response.content_blocks)
# [
#     {"type": "text", "text": "Here's a picture of a cat"},
#     {"type": "image", "base64": "...", "mime_type": "image/jpeg"},
# ]

### 4.2 推理
较新的模型能够执行多步推理以得出结论。这涉及将复杂问题分解为更小、更易于管理的步骤。

如果底层模型支持，您可以显示此推理过程，以更好地理解模型如何得出最终答案。

In [ ]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="qwen3:0.6b")
for chunk in model.stream("Why do parrots have colorful feathers?"):
    reasoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
    print(reasoning_steps if reasoning_steps else chunk.text)

In [ ]:
response = model.invoke("Why do parrots have colorful feathers?")
reasoning_steps = [b for b in response.content_blocks if b["type"] == "reasoning"]
print(" ".join(step["reasoning"] for step in reasoning_steps))

根据模型，有时可以指定其在推理上应付出的努力程度。同样，可以请求模型完全关闭推理。这可能以推理的类别“层级”（例如，'low' 或 'high'）或整数 token 预算的形式出现。